# TeleRAG: RAG over Family Documents

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

The data is a synthetic set of documents for one fictional family (the Beckers): insurance cards, passports, contracts, tax IDs. The goal is short, direct answers to lookup questions like "What's my insurance ID?".

In [1]:
import json
from pathlib import Path

import frontmatter
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load the Family Documents

Each document is a markdown file in `data/family-docs/` with frontmatter (`title`, `owner`, `doc_type`) and the document text as the body.

In [2]:
data_dir = Path("../data/family-docs")

documents = []
for path in sorted(data_dir.glob("*.md")):
    post = frontmatter.load(path)
    documents.append({
        "title": post.metadata.get("title"),
        "owner": post.metadata.get("owner"),
        "doc_type": post.metadata.get("doc_type"),
        "content": post.content,
        "filename": path.name,
    })

print(f"Loaded {len(documents)} documents")
documents[0]

Loaded 15 documents


{'title': 'Health insurance card - Anna Becker',
 'owner': 'Anna Becker',
 'doc_type': 'health_insurance',
 'content': "Insurer: Techniker Krankenkasse (TK)\nInsured person: Anna Becker, born 14.03.1986\nHealth insurance number (Versichertennummer): A847261935\nCard number: 80276 00184 2219\nCard valid until: 12/2028\nInsurer service hotline: 0800 285 8585\n\nMia Becker is co-insured through Anna's family insurance (Familienversicherung).",
 'filename': 'anna-health-insurance.md'}

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["title", "owner", "content"],
    keyword_fields=["owner", "doc_type", "filename"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You're TeleRAG, a Telegram assistant that looks up facts in a family's
personal documents (insurance cards, passports, contracts, tax letters).

Answer the QUESTION using only the CONTEXT documents:
- Reply in one or two short sentences, suitable for a chat message.
- Give the exact value (ID, number, date, phone) as written in the document,
  and name the document it came from.
- If the question doesn't say which family member it's about and the context
  has the value for several people, list each person's value on its own line.
- If the context doesn't contain the answer, say you couldn't find it in the
  documents. Never guess or make up numbers.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2, ensure_ascii=False)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("What's Mia's health insurance number?"))

Mia's health insurance number is M309584172 (from the health insurance card - Mia Becker).
